# <a id='toc1_'></a>[Renal Function - SADIA (no creatinine) | Recursive Feature Elimination - logistic regression with L2 penalty](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Renal Function - Infants (before year 1) | Recursive Feature Elimination - logistic regression with L2 penalty](#toc1_)    
  - [Configuration & data](#toc1_1_)    
  - [Modelling](#toc1_2_)    
    - [Recursive feature elimination with logistic regression with L2 penalty](#toc1_2_1_)    
  - [Results - Visualisatioins and significance testing](#toc1_3_)    
    - [Precision recall AUC by model feature count](#toc1_3_1_)    
    - [Anova - Precision-Recall AUC](#toc1_3_2_)    
    - [T-test - best vs 2nd best model with fewer features](#toc1_3_3_)    
    - [Balanced accuracy by model feature count](#toc1_3_4_)    
    - [Feature Coefficients of best model](#toc1_3_5_)    
    - [Confusion matrix of the best model](#toc1_3_6_)    
    - [Feature correlation heatmap of top features in the best model](#toc1_3_7_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Configuration & data](#toc0_)

In [ ]:
# logging setup
from puv.logging_setup import setup
setup()  # or DEBUG if you want more detail

import logging
logger = logging.getLogger(__name__)
logger.info("Logging initialized.")

In [ ]:
# Environment setup
from pathlib import Path

BASE = Path('/workspaces/CODESPACE/').resolve()

# Define paths for data and analysis directories & ensure key folders exist
PREP = BASE / "data" / "prep"
INT = BASE / "data" / "int" 
RES = BASE / "results" / "renal" / "sadia_nocrea"

for p in [PREP, INT]:
    p.mkdir(parents=True, exist_ok=True)

logger.info("Project base set to: %s", BASE)

In [ ]:
# Import dependencies
import numpy as np
import pandas as pd
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.anova import AnovaRM
from scipy.stats import randint, wilcoxon, loguniform
import matplotlib.pyplot as plt
import seaborn as sns


# Import custom functions
from puv.modelling import rfe_eval, select_best_k

plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 15, "axes.labelsize": 13,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
})


In [ ]:
# load data
df_renal = pd.read_excel(PREP / "renal_set_infants.xlsx")

# Uncomment to display the columns of the renal function dataset
# display(sorted(df_renal.columns.to_list()))

In [ ]:
# manual selection of features
df_sadia_nocrea = df_renal [[   'Patient number',
                                'Bilateral hydronephrosis',
                                'No hydronephrosis',
                                'Unilateral hydronephrosis',
                                'Bilateral dysplasia',
                                'No dysplasia',
                                'Unilateral dysplasia',
                                'Urinoma and/or Acites',
                                'Thick bladder wall',
                                'Unilateral VUR',
                                'Bilateral VUR',
                                'No VUR',
                                'Prenatal diagnosis',
                                'Renal function at last follow up',
                                ]].copy()


# uncomment to check the filtered dataframe
# display(sorted(df_sadia_nocrea.columns.to_list()))

## <a id='toc1_2_'></a>[Modelling](#toc0_)

### <a id='toc1_2_1_'></a>[Recursive feature elimination with logistic regression with L2 penalty](#toc0_)

In [ ]:
# drop the patient number identifier 
df_analysis = df_sadia_nocrea.copy()

# drop the patient number identifier and 'CKD5' columns - these are not relevant for this analysis
df_analysis.drop(columns=['Patient number'], inplace=True)

In [ ]:
pickle_path = INT / "renal_set_sadia_nocrea_rfe_lr_l2.pkl"

if pickle_path.exists():
    with pickle_path.open("rb") as f:
        rfe_evaluation_dict = pickle.load(f)
else:
    lr_l2 = LogisticRegression(
        penalty="l2",
        solver="liblinear",
        C=1.0,  # base value; nested CV will override via param_distributions
        class_weight="balanced",
        max_iter=5000,
        random_state=42,
    )

    param_distributions = {
        "clf__C": loguniform(1e-3, 1e3),
        "rfe__step": [1],
    }

    rfe_evaluation_dict = rfe_eval(
        df_raw=df_analysis,
        target_col="Renal function at last follow up",
        pos_label=1,  # 1 = CKD (minority); set explicitly, do not rely on default
        estimator=lr_l2,
        tuner="random",
        tune_hyperparams=True,
        n_iter=40,
        param_distributions=param_distributions,
        scoring="average_precision",
        show_progress=True,
        verbose=1,
        sklearn_verbose=0,
        cache_dir=str((BASE / ".model_cache").resolve()),
        gower_n_neighbors=3,
        gower_weights="distance",
        numeric_scaler=StandardScaler(),
        max_feature_eval=5,
        rfe_step=1,
    )

    with pickle_path.open("wb") as f:
        pickle.dump(rfe_evaluation_dict, f)

## <a id='toc1_3_'></a>[Results - Visualisatioins and significance testing](#toc0_)

In [ ]:
# extract results for the Precision-Recall AUC
summary = rfe_evaluation_dict["Summary"]
per_fold = rfe_evaluation_dict["PerFold"]
pr_auc_long = rfe_evaluation_dict["PR_AUC_ANOVA_Data"]

### <a id='toc1_3_1_'></a>[Precision recall AUC by model feature count](#toc0_)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# bar plot of mean balanced accuracy
ax.bar(summary["N_Features"],
       summary["PR_AUC_Mean"],
       yerr=summary["PR_AUC_SEM"],
       color="skyblue",
       edgecolor='black',
       alpha=0.8,
       capsize=4,
       label="Mean ± SEM")

# jittered fold-level points
xs, ys = [], []
for k, folds in per_fold.items():
    n = len(folds)
    # add small random horizontal "jitter" around k
    jitter = np.random.uniform(-0.15, 0.15, size=n)
    xs.extend(k + jitter)
    ys.extend([f["PR_AUC"] for f in folds])

ax.scatter(xs, ys, color="grey", alpha=0.5, zorder=3, label="Fold scores (jittered)")

# add mean (black) and ±SEM (red) labels on each bar
for x, mean, sem in zip(summary["N_Features"], summary["PR_AUC_Mean"], summary["PR_AUC_SEM"]):
    base_y = mean + sem + 0.1
    ax.text(x, base_y + 0.12, f"{mean:.2f}", ha="center", va="bottom", fontsize=11, color="black")
    ax.text(x, base_y, f"±{sem:.2f}", ha="center", va="bottom", fontsize=11, color="red")


ax.grid(False)
ax.set_ylim(0,1.5)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_xlabel("# features")
ax.set_ylabel("PR-AUC")
ax.set_title("PR-AUC by model feature count")
ax.legend(loc="lower center", framealpha=1.0)
plt.tight_layout()

plt.savefig(RES / "renal_rfe_lr-l2_pr_auc.png", dpi=300)
plt.show()



### <a id='toc1_3_2_'></a>[Anova - Precision-Recall AUC](#toc0_)

In [ ]:
anova_df = pr_auc_long.rename(columns={"n_features": "n_features", "fold": "fold", "score": "PR_AUC"})
anova_model = AnovaRM(anova_df, depvar="PR_AUC", subject="fold", within=["n_features"]).fit()
logger.info("PR-AUC repeated-measures ANOVA:\n%s", anova_model.summary())

### <a id='toc1_3_3_'></a>[T-test - best vs 2nd best model with fewer features](#toc0_)

In [ ]:
# Naive empirical-best feature count (argmax mean PR-AUC), kept only for the comparison log below.
data_driven_best_k = int(summary.loc[summary["PR_AUC_Mean"].idxmax(), "N_Features"])

# Principled model selection (see puv.modelling.select_best_k):
#   1) repeated-measures ANOVA of PR-AUC across feature counts (folds as within-subject factor);
#      if no subset size differs significantly, default to the single-feature model;
#   2) otherwise compare the highest-mean subset with the best-performing smaller subset via a
#      paired Wilcoxon signed-rank test, keeping the larger set only if significantly better.
best_k = select_best_k(rfe_evaluation_dict)

# Log how the principled choice compares to the naive empirical-best feature count.
if data_driven_best_k == best_k:
    logger.info("Selected k=%d equals the empirical-best feature count; nothing larger to compare.", best_k)
else:
    pr_auc_sel = np.array([f["PR_AUC"] for f in per_fold[best_k]])
    pr_auc_best = np.array([f["PR_AUC"] for f in per_fold[data_driven_best_k]])
    diffs = pr_auc_sel - pr_auc_best
    if np.allclose(diffs, 0):
        logger.info("PR-AUC identical for k=%d and k=%d; Wilcoxon trivial (p=1).",
                    best_k, data_driven_best_k)
    else:
        mask = ~np.isclose(diffs, 0)
        stat, p_val = wilcoxon(pr_auc_sel[mask], pr_auc_best[mask],
                               zero_method="wilcox", alternative="two-sided")
        logger.info("Wilcoxon signed-rank (selected k=%d vs empirical-best k=%d): W=%.3f, p=%.4f",
                    best_k, data_driven_best_k, stat, p_val)

### <a id='toc1_3_4_'></a>[Balanced accuracy by model feature count](#toc0_)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# bar plot of mean balanced accuracy
ax.bar(summary["N_Features"],
       summary["BalancedAccuracy_Mean"],
       yerr=summary["BalancedAccuracy_SEM"],
       color="skyblue",
        edgecolor='black',
       alpha=0.8,
       capsize=4,
       label="Mean ± SEM")

# jittered fold-level points overlaid
xs, ys = [], []
for k, folds in per_fold.items():
    jitter = np.random.uniform(-0.25, 0.25, size=len(folds))
    xs.extend(k + jitter)
    ys.extend([f["BalancedAccuracy"] for f in folds])

ax.scatter(xs, ys, color="gray", alpha=0.5, zorder=3, label="Fold scores")

# add mean (black) and ±SEM (red) labels on each bar
for x, mean, sem in zip(summary["N_Features"], summary["BalancedAccuracy_Mean"], summary["BalancedAccuracy_SEM"]):
    base_y = mean + sem + 0.1
    ax.text(x, base_y + 0.12, f"{mean:.2f}", ha="center", va="bottom", fontsize=11, color="black")
    ax.text(x, base_y, f"±{sem:.2f}", ha="center", va="bottom", fontsize=11, color="red")


ax.grid(False)
ax.set_ylim(0, 1.5)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_xlabel("# features")
ax.set_ylabel("Balanced accuracy")
ax.set_title("Balanced accuracy by model feature count")
ax.legend()
plt.tight_layout()

plt.savefig(RES / "renal_rfe_lr-l2_balanced_accuracy.png", dpi=300)
plt.show()


### <a id='toc1_3_5_'></a>[Feature Coefficients of best model](#toc0_)

In [ ]:
# Feature Coefficent plot for best model
imp_list = rfe_evaluation_dict["FeatureImportances"][best_k]
if not imp_list:
    logger.info("No feature coefficient data recorded for k=%d.", best_k)
else:

    all_features = sorted({name for fold in imp_list for name in fold})
    mat = np.vstack([[fold.get(name, np.nan) for name in all_features] for fold in imp_list])

    mean_imp = np.nanmean(mat, axis=0)
    sem_imp = np.nanstd(mat, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(mat), axis=0))

        # --- Rank features by mean importance (descending) ---
    order = np.argsort(mean_imp)  # highest → lowest

    all_features = [all_features[i] for i in order]
    mean_imp = mean_imp[order]
    sem_imp = sem_imp[order]
    mat = mat[:, order]

    fig, ax = plt.subplots(figsize=(max(6, 0.4 * len(all_features)), 4))

    # --- Horizontal bar plot ---
    ax.barh(
        all_features,
        mean_imp,
        xerr=sem_imp,
        color="skyblue",
        edgecolor="black",
        alpha=0.9,
        capsize=4,
    )

    # --- Add jittered raw data points ---
    jitter_strength = 0.1
    for i, feature in enumerate(all_features):
        raw_vals = mat[:, i]

        # remove nans
        raw_vals = raw_vals[~np.isnan(raw_vals)]

        # jitter around y = i
        y_positions = i + np.random.uniform(-jitter_strength, jitter_strength, size=len(raw_vals))

        ax.scatter(
            raw_vals,
            y_positions,
            color="grey",
            s=20,
            alpha=0.65,
            zorder=3
        )
        
    
    # --- Expand x-limit to prevent label clipping ---
    x_min = np.nanmin(mean_imp)
    x_max = np.nanmax(mean_imp)
    ax.set_xlim(min (0, x_min)* 3.15, x_max * 5.5)

    # --- Add value labels at end of bars ---
    # --- Add mean (black) and SEM (red) labels ---
    offset = 0.33 * ax.get_xlim()[1]
    sem_offset = 0.55 * ax.get_xlim()[1]   # slightly further right


    for i, val in enumerate(mean_imp):
        ax.text(
            val + offset,
            i,
            f"{val:.2f}",
            va="center",
            ha="left",
            fontsize=11,
            color="black"
        )

        # SEM value (red)
        ax.text(
            val + sem_offset,
            i,
            f"±{sem_imp[i]:.2f}",
            va="center",
            ha="left",
            fontsize=11,
            color="red"
        )


    ax.grid(False)
    ax.set_xlabel("coefficient (mean ± SEM)")
    ax.set_title(f"Feature coefficient of best model (feature count per fold = {best_k})")
    plt.tight_layout()
    plt.savefig(RES / "renal_rfe_lr-l2_feature_coefficients.png", dpi=300)
    plt.show()


### <a id='toc1_3_6_'></a>[Confusion matrix of the best model](#toc0_)

In [ ]:
cms = rfe_evaluation_dict["ConfusionMatrices"][best_k]

# Build cube of shape (fold, 2, 2)
cube = np.array([
    [[cm["TN"], cm["FP"]],
     [cm["FN"], cm["TP"]]]
    for cm in cms
])

mean_cm = cube.mean(axis=0)
sem_cm  = cube.std(axis=0, ddof=1) / np.sqrt(cube.shape[0])
std_cm  = cube.std(axis=0, ddof=1)

# --- ROW NORMALIZATION ---
row_sums = mean_cm.sum(axis=1, keepdims=True)
mean_pct = mean_cm / row_sums * 100
sem_pct  = sem_cm  / row_sums * 100
std_pct  = std_cm  / row_sums * 100

fig, ax = plt.subplots(figsize=(5.5, 5.5))
im = ax.imshow(mean_pct, cmap="Blues", vmin=0, vmax=100)

ax.set_xticks([0, 1]); 
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred 0", "Pred 1"])
ax.set_yticklabels(["True 0", "True 1"])

ax.set_title(f"Row-normalized confusion matrix of the best model", pad=40)

# --- Text annotations with selective white coloring ---
for i in range(2):
    for j in range(2):

        # White text for TN (0,0) and TP (1,1)
        if (i == 0 and j == 0) or (i == 1 and j == 1):
            color = "white"
        else:
            color = "black"

        ax.text(
            j, i,
            f"{mean_pct[i, j]:.1f}%\n±SEM {sem_pct[i, j]:.1f}%\n±SD {std_pct[i, j]:.1f}%",
            ha="center", va="center", color=color, fontweight="light"
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.1)
cbar.set_label("% (row-normalized)")
ax.grid(False)
plt.tight_layout()

plt.savefig(RES / "renal_rfe_lr-l2_confusion_matrix_row_normalized.png", dpi=300)
plt.show()

### <a id='toc1_3_7_'></a>[Feature correlation heatmap of top features in the best model](#toc0_)

In [ ]:
best_selected = rfe_evaluation_dict["Selected_Features"][best_k]
unique_feature_names= list({item for row in best_selected for item in row})
df_subset = df_analysis[unique_feature_names]

# Compute Spearman correlation
corr = df_subset.corr(method="spearman")

n = corr.shape[0]
fig_size = max(6, min(1.2 * n, 18))

fig, ax = plt.subplots(figsize=(fig_size, fig_size))

# Draw heatmap **without** seaborn's annot
sns.heatmap(
    corr,
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8},
    annot=False,
    ax=ax,
)

# Manually add text in EVERY cell
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        val = corr.iloc[i, j]
        if np.isnan(val):
            continue

        # Black for small |ρ|, white for larger |ρ|
        txt_color = "black" if abs(val) < 0.25 else "white"

        ax.text(
            j + 0.5,          # x position (cell centre)
            i + 0.5,          # y position (cell centre)
            f"{val:.2f}",
            ha="center",
            va="center",
            color=txt_color,
            fontsize=12,
            zorder=5,
        )

ax.set_title("Feature correlation heatmap", fontsize=22, pad=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=15)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=15)

plt.tight_layout()
plt.savefig(RES / "renal_rfe_lr-l2_feature_correlation_heatmap.png", dpi=300)
plt.show()
